# **Instant Voice Cloning using Sarvam AI API**

This notebook creates a saved cloned voice from a short reference clip, generates one line of speech, then streams a second line. Instant Voice Cloning is in **beta**.

Only clone a voice you have the right to use. See [Commercial Licensing](https://docs.sarvam.ai/api/getting-started/commercial-licensing).

## **Prerequisites**

Before running this notebook, ensure you have the following:

- Python 3.9 or higher
- `sarvamai` ≥ `0.1.37`
- A 10-15 second, single-speaker reference clip named `reference.wav` in the working directory. See [Prepare Reference Audio](https://docs.sarvam.ai/api/api-guides-tutorials/voice-cloning/how-to/prepare-reference-audio).

You can install the required package using pip:

In [ ]:
! pip install -Uqq "sarvamai>=0.1.37"

## **Import Required Libraries**

First, let's import all the necessary libraries.

In [ ]:
import base64

from sarvamai import SarvamAI

### **1. Set up your API key**

To use Instant Voice Cloning, you need an API subscription key. Follow these steps:

1. **Obtain your API key**: If you don’t have an API key, sign up on the [Sarvam AI Dashboard](https://dashboard.sarvam.ai/key-management) to get one.
2. **Replace the placeholder key**: In the code below, replace `"YOUR_SARVAM_API_KEY"` with your actual API key.

In [ ]:
SARVAM_API_KEY = "YOUR_SARVAM_API_KEY"

In [ ]:
client = SarvamAI(api_subscription_key=SARVAM_API_KEY)

## **Understanding the Parameters**

### Create a saved voice (`create_voice`)

| Parameter | Type | Description |
|-----------|------|-------------|
| `file` | file | Reference clip. Recommended 10-15 seconds, single speaker, WAV. Max 50 MB. |
| `name` | string | Display name for the saved voice |
| `language` | string | BCP-47 language of the reference clip (e.g. `en-IN`) |

### Generate speech (`convert` / `convert_stream`)

| Parameter | Type | Description |
|-----------|------|-------------|
| `voice_id` | string | Saved voice from `create_voice`. Send exactly one of `voice_id` or `ref_audio`. |
| `text` | string | Text to speak. Max 1000 characters on REST and HTTP stream. |
| `language_code` | string | Optional. Omit to detect the language from `text`. Pass a code when pronunciation has to be exact. See [Supported languages](https://docs.sarvam.ai/api/api-guides-tutorials/voice-cloning/supported-languages). |

`convert` returns JSON with base64 audio (default WAV). `convert_stream` returns binary chunks as each sentence is ready (default MP3).

### **2. Create a saved voice**

Put `reference.wav` next to this notebook, then create the voice once and keep `voice_id`.

In [ ]:
with open("reference.wav", "rb") as f:
    created = client.voice_cloning.create_voice(
        file=f,
        name="my-voice",
        language="en-IN",
    )
voice_id = created.data.voice_id
print(f"Created voice: {voice_id}")

### **3. Generate one line (one-shot)**

`language_code` is omitted so the service detects the language from `text`. Decode `audio_b64` or `audio` and write a WAV.

In [ ]:
oneshot = client.voice_cloning.convert(
    voice_id=voice_id,
    text="नमस्ते, मैं आपकी कैसे मदद कर सकता हूँ?",
)
with open("output.wav", "wb") as out:
    out.write(base64.b64decode(oneshot.audio_b64 or oneshot.audio))
print("Wrote output.wav")

### **4. Stream a second line**

HTTP streaming returns binary audio as soon as the first sentence is ready. The stream default codec is MP3.

In [ ]:
with open("output_stream.mp3", "wb") as out:
    for chunk in client.voice_cloning.convert_stream(
        voice_id=voice_id,
        text="यह एक स्ट्रीमिंग टेस्ट है। ऑडियो तुरंत मिलना शुरू हो जाना चाहिए।",
    ):
        out.write(chunk)
print("Wrote output_stream.mp3")

## **Output**

After running the notebook you will have:

- a saved `voice_id` you can reuse on later generate calls
- `output.wav` from the one-shot call
- `output_stream.mp3` from the HTTP stream

Both files should sound like the speaker in `reference.wav`. The saved voice also appears under Studio **My voices**.

## **Error Handling**

You may encounter these errors while using the API:

| Error Code | HTTP Status | Cause | Solution |
|------------|-------------|-------|----------|
| `invalid_api_key_error` | 403 Forbidden | Invalid API key | Use a valid API key from the [Sarvam AI Dashboard](https://dashboard.sarvam.ai/) |
| `insufficient_quota_error` | 402 / 429 | Credits or quota | Check usage and retry with backoff on 429 |
| `invalid_request_error` | 400 Bad Request | Both `voice_id` and `ref_audio`, a clip that is too long, or romanized Indic / Urdu text | Send exactly one of `voice_id` or `ref_audio`; use native script |
| `rate_limit_exceeded_error` | 429 Too Many Requests | Rate limit exceeded | Retry with exponential backoff |
| `internal_server_error` | 500 Internal Server Error | Issue on servers | Try again later. If persistent, contact support |

Retry `429`, `500`, `502`, `503`, and `504`. Do not retry `400`, `401`, `402`, `403`, `404`, or `422`. Log the `request_id` from any failed response.

## **Conclusion**

This notebook walks through creating a saved cloned voice and generating speech with it, one-shot and streamed, using the same `sarvamai` client shape as Bulbul Text-to-Speech.

### **Additional Resources**

- **Docs cookbook**: [Instant Voice Cloning](https://docs.sarvam.ai/api/cookbook/guides/instant-voice-cloning)
- **API overview**: [Instant Voice Cloning](https://docs.sarvam.ai/api/api-guides-tutorials/voice-cloning/overview)
- **How-tos**: [Prepare Reference Audio](https://docs.sarvam.ai/api/api-guides-tutorials/voice-cloning/how-to/prepare-reference-audio) · [HTTP Streaming](https://docs.sarvam.ai/api/api-guides-tutorials/voice-cloning/how-to/http-streaming) · [Supported languages](https://docs.sarvam.ai/api/api-guides-tutorials/voice-cloning/supported-languages)
- **Community**: [Join the Discord Community](https://discord.com/invite/5rAsykttcs)

---

### **Final Notes**

- Keep your API key secure.
- Create the voice once, then reuse `voice_id` on every generate call.
- Only clone a voice you have the right to use.
- Omit `language_code` for typical text; pass it when pronunciation has to be exact.

**Keep Building!**